In [1]:
import os, math, json, random
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Optional

try:
    from rdkit import Chem
    from rdkit.Chem import DataStructs
    from rdkit import DataStructs as ds
except Exception as e:
    raise RuntimeError("RDKit is required for Tanimoto similarity calculation.") from e

# =============================================================================
# Configuration
# =============================================================================

# Data paths
MERGED_DATA_CSV = "../dataset/training_pool/KinSPEC_training_pool_merged_4_datasets_deduplicated_with_smiles.csv"
VSDS_CSV = "../dataset/training_pool/VSDS_truedecoy_integrated_data_with_protein_seq.csv"
OUTDIR = "../dataset/spectrum_data/"
os.makedirs(OUTDIR, exist_ok=True)

# Column configuration
FP_COL = "Morgan"
KLIFS_COL = "KLIFS"
LABEL_COL = "unified_bio_label"  # assumed column name; adjust as needed

# Curriculum parameters
WINDOW = 0.4
STEP = 0.1
BANDS = [(start, start + WINDOW) for start in np.arange(0, 1.0, STEP) if start + WINDOW <= 1.0 + 1e-6]
PER_BAND = 100000
TOPK = 1
SEED = 42

# =============================================================================
# Utilities
# =============================================================================

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)

def ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)

def parse_fingerprint(fp_str: str) -> Optional[DataStructs.ExplicitBitVect]:
    """Parse fingerprint from various string formats."""
    if fp_str is None or pd.isna(fp_str):
        return None
    fp_str = str(fp_str).strip()
    if not fp_str:
        return None
    bv = DataStructs.ExplicitBitVect(4096)
    try:
        if fp_str.startswith(('0x', '0X')):
            hex_str = fp_str[2:]
            if len(hex_str) == 0:
                return None
            binary_str = bin(int(hex_str, 16))[2:].zfill(4096)
            for i, bit in enumerate(binary_str[:4096]):
                if bit == '1':
                    bv.SetBit(i)
        elif ',' in fp_str:
            bits = [int(x.strip()) for x in fp_str.split(',')]
            for i, bit in enumerate(bits[:4096]):
                if bit == 1:
                    bv.SetBit(i)
        elif all(c in '01 \t\n' for c in fp_str):
            clean_str = fp_str.replace(' ', '').replace('\t', '').replace('\n', '')
            for i, bit in enumerate(clean_str[:4096]):
                if bit == '1':
                    bv.SetBit(i)
        else:
            print(f"Warning: Unsupported fingerprint format: {fp_str[:50]}...")
            return None
    except Exception as e:
        print(f"Warning: Could not parse fingerprint {fp_str[:50]}...: {e}")
        return None
    return bv

# =============================================================================
# Data Loading
# =============================================================================

def load_merged_data(csv_path: str, fp_col: str) -> pd.DataFrame:
    """Load the deduplicated merged data"""
    print(f"\n📂 Loading merged data from: {csv_path}")
    
    try:
        df = pd.read_csv(csv_path)
        print(f"  ✓ Loaded {len(df)} records")
        print(f"  ✓ Columns: {list(df.columns)}")
        
        # Check required columns
        if fp_col not in df.columns:
            raise ValueError(f"Missing required column: {fp_col}")
        
        # Check and print data statistics
        print(f"\n📊 Data Statistics:")
        print(f"  Total records: {len(df)}")
        
        if LABEL_COL in df.columns:
            labeled = df[df[LABEL_COL].notna()]
            actives = labeled[labeled[LABEL_COL] == 1]
            inactives = labeled[labeled[LABEL_COL] == 0]
            unlabeled = df[df[LABEL_COL].isna()]
            
            print(f"  Labeled records: {len(labeled)}")
            print(f"    - Active: {len(actives)} ({len(actives)/len(df)*100:.1f}%)")
            print(f"    - Inactive: {len(inactives)} ({len(inactives)/len(df)*100:.1f}%)")
            print(f"  Unlabeled records: {len(unlabeled)} ({len(unlabeled)/len(df)*100:.1f}%)")
        
        return df
        
    except Exception as e:
        raise RuntimeError(f"Failed to load merged data: {e}")

def load_vsds_data(csv_path: str, fp_col: str) -> List[DataStructs.ExplicitBitVect]:
    """Load VSDS data and parse fingerprints"""
    print(f"\n📂 Loading VSDS data from: {csv_path}")
    
    try:
        df = pd.read_csv(csv_path)
        print(f"  ✓ Loaded {len(df)} VSDS records")
        
        if fp_col not in df.columns:
            raise ValueError(f"Missing required column: {fp_col}")
        
        # Parse fingerprints
        vsds_fps = []
        failed = 0
        for fp_str in df[fp_col]:
            fp = parse_fingerprint(fp_str)
            if fp is not None:
                vsds_fps.append(fp)
            else:
                failed += 1
        
        print(f"  ✓ Parsed {len(vsds_fps)} fingerprints")
        if failed > 0:
            print(f"  ⚠️  Failed to parse {failed} fingerprints")
        
        return vsds_fps
        
    except Exception as e:
        raise RuntimeError(f"Failed to load VSDS data: {e}")

# =============================================================================
# Similarity Computation (Using Bulk Tanimoto)
# =============================================================================

def compute_similarities_bulk(merged_df: pd.DataFrame, 
                              vsds_fps: List[DataStructs.ExplicitBitVect],
                              fp_col: str,
                              topk: int = 1,
                              batch_size: int = 1000) -> pd.DataFrame:
    """
    Use Bulk Tanimoto to calculate similarity between the merged data and VSDS
    batch_size: number of merged molecules processed per batch
    """
    print(f"\n🔬 Computing similarities using Bulk Tanimoto: {len(merged_df)} merged × {len(vsds_fps)} VSDS...")
    
    # Parse fingerprints from the merged data
    print("  Parsing merged data fingerprints...")
    merged_fps = []
    valid_indices = []
    failed = 0
    
    for idx, fp_str in enumerate(merged_df[fp_col]):
        fp = parse_fingerprint(fp_str)
        if fp is not None:
            merged_fps.append(fp)
            valid_indices.append(idx)
        else:
            failed += 1
    
    print(f"  ✓ Parsed {len(merged_fps)} fingerprints")
    if failed > 0:
        print(f"  ⚠️  Failed to parse {failed} fingerprints")
    
    # Keep only records whose fingerprints were parsed successfully
    merged_df = merged_df.iloc[valid_indices].reset_index(drop=True)
    
    # Calculate similarity using Bulk Tanimoto
    print(f"  Computing Bulk Tanimoto similarities (batch_size={batch_size})...")
    
    similarities = []
    n_merged = len(merged_fps)
    n_batches = (n_merged + batch_size - 1) // batch_size
    
    for batch_idx in range(n_batches):
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, n_merged)
        batch_fps = merged_fps[start_idx:end_idx]
        
        if batch_idx % 10 == 0:
            print(f"    Progress: Batch {batch_idx + 1}/{n_batches} ({start_idx}/{n_merged}, {start_idx/n_merged*100:.1f}%)")
        
        # For each molecule in the current batch, calculate similarity against all VSDS molecules
        for merged_fp in batch_fps:
            # Use BulkTanimotoSimilarity to calculate similarity against all VSDS molecules at once
            sims = ds.BulkTanimotoSimilarity(merged_fp, vsds_fps)
            
            # Filter NaN values
            valid_sims = [s for s in sims if not math.isnan(s)]
            
            if not valid_sims:
                max_sim = 0.0
                topk_avg = 0.0
            else:
                sims_sorted = sorted(valid_sims, reverse=True)
                max_sim = sims_sorted[0]
                topk_avg = float(np.mean(sims_sorted[:min(topk, len(sims_sorted))]))
            
            similarities.append({
                'max_sim_vsds': max_sim,
                f'top{topk}_avg_sim_vsds': topk_avg
            })
    
    # Add the similarity column to the dataframe
    sim_df = pd.DataFrame(similarities)
    result_df = pd.concat([merged_df.reset_index(drop=True), sim_df], axis=1)
    
    print(f"  ✅ Completed similarity calculations")
    print(f"  Similarity range: [{result_df['max_sim_vsds'].min():.3f}, {result_df['max_sim_vsds'].max():.3f}]")
    print(f"  Mean similarity: {result_df['max_sim_vsds'].mean():.3f}")
    
    return result_df

# =============================================================================
# Band Splitting
# =============================================================================

def create_band_splits(data_df: pd.DataFrame,
                      bands: List[Tuple[float, float]],
                      per_band: int,
                      label_col: str,
                      fp_col: str,
                      klifs_col: str,
                      seed: int = 42) -> Dict[str, pd.DataFrame]:
    """
    Divide data by similarity band, ensure unique (Morgan, KLIFS) pairs, and control the active-molecule ratio
    """
    rng = np.random.default_rng(seed)
    splits = {}
    TARGET_ACTIVE = 5000

    for lo, hi in bands:
        band_name = f"band_{lo:.2f}_{hi:.2f}"
        
        # Filter records that fall within the current band
        band_df = data_df[
            (data_df['max_sim_vsds'] >= lo) &
            (data_df['max_sim_vsds'] < hi)
        ].copy()
        
        print(f"\n  📊 {band_name}")
        print(f"    Initial records: {len(band_df)}")
        
        if len(band_df) == 0:
            splits[band_name] = pd.DataFrame()
            print(f"    ⚠️  No candidates in this band")
            continue
        
        # Core step: deduplicate by (Morgan, KLIFS)
        if fp_col in band_df.columns and klifs_col in band_df.columns:
            before_dedup = len(band_df)
            band_df['_combo_key'] = (band_df[fp_col].astype(str) + '|||' + 
                                     band_df[klifs_col].astype(str))
            
            grouped = band_df.groupby('_combo_key', group_keys=False)
            
            def select_best_record(group):
                """Prioritise records that are labelled and active"""
                if label_col not in group.columns:
                    return group.iloc[0:1]
                
                # Labelled records
                labeled = group[group[label_col].notna()]
                if len(labeled) == 0:
                    return group.iloc[0:1]
                
                # Prioritise active molecules
                actives = labeled[labeled[label_col] == 1]
                if len(actives) > 0:
                    return actives.iloc[0:1]
                
                # Then return inactive molecules
                return labeled.iloc[0:1]
            
            band_df = grouped.apply(select_best_record).reset_index(drop=True)
            band_df = band_df.drop(columns=['_combo_key'])
            
            removed_dup = before_dedup - len(band_df)
            print(f"    After (Morgan, KLIFS) dedup: {len(band_df)} (removed {removed_dup})")
        
        total = len(band_df)
        
        # Summarise label distribution
        if label_col in band_df.columns:
            labeled = band_df[band_df[label_col].notna()]
            unlabeled = band_df[band_df[label_col].isna()]
            actives = labeled[labeled[label_col] == 1]
            inactives = labeled[labeled[label_col] == 0]
            print(f"    Labeled: {len(labeled)} (Active: {len(actives)}, Inactive: {len(inactives)})")
            print(f"    Unlabeled: {len(unlabeled)}")
        else:
            labeled = band_df
            unlabeled = pd.DataFrame()
            actives = pd.DataFrame()
            inactives = pd.DataFrame()
        
        # Sampling strategy
        if total >= per_band:
            picks = []
            
            # 1. Select active molecules
            if len(actives) < TARGET_ACTIVE:
                picks.append(actives)
                need = per_band - len(actives)
            else:
                picks.append(actives.sample(n=TARGET_ACTIVE, random_state=seed))
                need = per_band - TARGET_ACTIVE
            
            # 2. Add inactive molecules
            if need > 0:
                take_i = min(need, len(inactives))
                if take_i > 0:
                    picks.append(inactives.sample(n=take_i, random_state=seed))
                    need -= take_i
            
            # 3. Add unlabelled molecules
            if need > 0 and len(unlabeled) > 0:
                take_u = min(need, len(unlabeled))
                picks.append(unlabeled.sample(n=take_u, random_state=seed))
                need -= take_u
            
            # 4. If still insufficient, randomly select from the remaining records
            picked = pd.concat(picks, axis=0) if picks else pd.DataFrame(columns=band_df.columns)
            
            if len(picked) < per_band and len(band_df) > len(picked):
                remaining = band_df.loc[~band_df.index.isin(picked.index)]
                take_r = min(per_band - len(picked), len(remaining))
                if take_r > 0:
                    picked = pd.concat([picked, remaining.sample(n=take_r, random_state=seed)], axis=0)
            
            # Shuffle the order
            selected = picked.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        else:
            # If the number of candidates is below per_band, keep all candidates
            selected = band_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        
        # Final statistics
        if label_col in selected.columns:
            final_actives = int(selected[label_col].eq(1).sum())
            final_total = len(selected)
            final_ratio = final_actives / final_total if final_total > 0 else 0.0
            print(f"    ✅ Final: {final_total} records | Active: {final_actives} ({final_ratio:.1%})")
        else:
            print(f"    ✅ Final: {len(selected)} records")
        
        splits[band_name] = selected

    return splits

# =============================================================================
# Main Execution
# =============================================================================

def main():
    set_seed(SEED)
    ensure_dir(OUTDIR)

    print("=" * 80)
    print("🧬 VSDS CURRICULUM BUILDER - extract bands from preprocessed data (Bulk Tanimoto)")
    print("=" * 80)
    print(f"Configuration:")
    print(f"  📁 Merged data: {MERGED_DATA_CSV}")
    print(f"  📁 VSDS data: {VSDS_CSV}")
    print(f"  🔬 Fingerprint column: {FP_COL}")
    print(f"  🏷️  KLIFS column: {KLIFS_COL}")
    print(f"  📊 Label column: {LABEL_COL}")
    print(f"  📂 Output directory: {OUTDIR}")

    # 1. Load data
    print("\n" + "=" * 60)
    print("1. LOADING DATA")
    print("=" * 60)
    
    merged_df = load_merged_data(MERGED_DATA_CSV, FP_COL)
    vsds_fps = load_vsds_data(VSDS_CSV, FP_COL)

    # 2. Compute similarities using Bulk Tanimoto
    print("\n" + "=" * 60)
    print("2. COMPUTING SIMILARITIES (Bulk Tanimoto)")
    print("=" * 60)
    
    data_with_sim = compute_similarities_bulk(
        merged_df, 
        vsds_fps, 
        FP_COL, 
        topk=TOPK,
        batch_size=50000  # adjust this value to balance memory usage and speed
    )
    
    # Save the full data with similarity values
    full_data_path = os.path.join(OUTDIR, "merged_data_with_similarities.csv")
    data_with_sim.to_csv(full_data_path, index=False)
    print(f"\n💾 Saved full data with similarities to: {full_data_path}")

    # 3. Create band splits
    print("\n" + "=" * 60)
    print("3. CREATING BAND SPLITS")
    print("=" * 60)
    
    band_splits = create_band_splits(
        data_with_sim,
        BANDS,
        PER_BAND,
        LABEL_COL,
        FP_COL,
        KLIFS_COL,
        seed=SEED
    )

    # 4. Save band splits
    print("\n" + "=" * 60)
    print("4. SAVING BAND SPLITS")
    print("=" * 60)
    
    for band_name, band_df in band_splits.items():
        if len(band_df) == 0:
            print(f"  ⚠️  Skipping empty band: {band_name}")
            continue
        
        split_dir = os.path.join(OUTDIR, band_name)
        ensure_dir(split_dir)
        
        # Save training data
        train_path = os.path.join(split_dir, "train.csv")
        band_df.to_csv(train_path, index=False)
        
        print(f"  💾 {band_name}: {len(band_df)} records")
        
        # Statistics
        if LABEL_COL in band_df.columns:
            total = len(band_df)
            actives = band_df[LABEL_COL].sum(skipna=True)
            active_ratio = actives / total if total > 0 else 0
            print(f"    🔎 Active molecules: {int(actives)}/{total} ({active_ratio:.2%})")
        
        # Save summary
        split_summary = {
            "total_records": len(band_df),
            "band_range": [float(band_name.split('_')[1]), float(band_name.split('_')[2])],
            "columns": list(band_df.columns)
        }
        
        if LABEL_COL in band_df.columns:
            split_summary["active_count"] = int(band_df[LABEL_COL].sum(skipna=True))
            split_summary["active_ratio"] = float(active_ratio)
        
        summary_path = os.path.join(split_dir, "split_summary.json")
        with open(summary_path, 'w') as f:
            json.dump(split_summary, f, indent=2)

    # 5. Overall summary
    print(f"\n" + "=" * 60)
    print("✅ COMPLETED!")
    print("=" * 60)
    print(f"📊 Summary:")
    print(f"  Total input records: {len(merged_df)}")
    print(f"  Records with valid fingerprints: {len(data_with_sim)}")
    print(f"  Number of bands created: {len([b for b in band_splits.values() if len(b) > 0])}")
    
    total_band_records = sum(len(b) for b in band_splits.values())
    print(f"  Total records in all bands: {total_band_records}")
    
    if LABEL_COL in data_with_sim.columns:
        total_actives = sum(int(b[LABEL_COL].sum(skipna=True)) for b in band_splits.values() if len(b) > 0)
        overall_ratio = total_actives / total_band_records if total_band_records > 0 else 0
        print(f"  Total active molecules across bands: {total_actives} ({overall_ratio:.1%})")
    
    print(f"\n✅ (Morgan, KLIFS) pairs have been deduplicated within each band!")
    print(f"⚡ Used Bulk Tanimoto to accelerate similarity calculation!")
    print(f"📁 All results saved to: {OUTDIR}")
    
    return True

if __name__ == "__main__":
    main()

🧬 VSDS CURRICULUM BUILDER - extract bands from preprocessed data (Bulk Tanimoto)
Configuration:
  📁 Merged data: ../dataset/training_pool/KinSPEC_training_pool_merged_4_datasets_deduplicated_with_smiles.csv
  📁 VSDS data: ../dataset/training_pool/VSDS_truedecoy_integrated_data_with_protein_seq.csv
  🔬 Fingerprint column: Morgan
  🏷️  KLIFS column: KLIFS
  📊 Label column: unified_bio_label
  📂 Output directory: ../dataset/spectrum_data/

1. LOADING DATA

📂 Loading merged data from: ../dataset/training_pool/KinSPEC_training_pool_merged_4_datasets_deduplicated_with_smiles.csv
  ✓ Loaded 692704 records
  ✓ Columns: ['Morgan', 'KLIFS', 'unified_bio_label', 'source']

📊 Data Statistics:
  Total records: 692704
  Labeled records: 692704
    - Active: 29708 (4.3%)
    - Inactive: 662996 (95.7%)
  Unlabeled records: 0 (0.0%)

📂 Loading VSDS data from: ../dataset/training_pool/VSDS_truedecoy_integrated_data_with_protein_seq.csv
  ✓ Loaded 49706 VSDS records
  ✓ Parsed 49706 fingerprints

2. COMP

/tmp/ipykernel_2132199/11608396.py:292: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  band_df = grouped.apply(select_best_record).reset_index(drop=True)


    After (Morgan, KLIFS) dedup: 318799 (removed 0)
    Labeled: 318799 (Active: 2039, Inactive: 316760)
    Unlabeled: 0
    ✅ Final: 100000 records | Active: 2039 (2.0%)

  📊 band_0.10_0.50
    Initial records: 419347


/tmp/ipykernel_2132199/11608396.py:292: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  band_df = grouped.apply(select_best_record).reset_index(drop=True)


    After (Morgan, KLIFS) dedup: 419347 (removed 0)
    Labeled: 419347 (Active: 5161, Inactive: 414186)
    Unlabeled: 0
    ✅ Final: 100000 records | Active: 5000 (5.0%)

  📊 band_0.20_0.60
    Initial records: 482863


...

    After (Morgan, KLIFS) dedup: 272815 (removed 0)
    Labeled: 272815 (Active: 16199, Inactive: 256616)
    Unlabeled: 0
    ✅ Final: 100000 records | Active: 5000 (5.0%)

  📊 band_0.50_0.90
    Initial records: 187969


/tmp/ipykernel_2132199/11608396.py:292: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  band_df = grouped.apply(select_best_record).reset_index(drop=True)


    After (Morgan, KLIFS) dedup: 187969 (removed 0)
    Labeled: 187969 (Active: 15487, Inactive: 172482)
    Unlabeled: 0
    ✅ Final: 100000 records | Active: 5000 (5.0%)

  📊 band_0.60_1.00
    Initial records: 136197


/tmp/ipykernel_2132199/11608396.py:292: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  band_df = grouped.apply(select_best_record).reset_index(drop=True)


    After (Morgan, KLIFS) dedup: 136197 (removed 0)
    Labeled: 136197 (Active: 13145, Inactive: 123052)
    Unlabeled: 0
    ✅ Final: 100000 records | Active: 5000 (5.0%)

4. SAVING BAND SPLITS
  💾 band_0.00_0.40: 100000 records
    🔎 Active molecules: 2039/100000 (2.04%)
  💾 band_0.10_0.50: 100000 records
    🔎 Active molecules: 5000/100000 (5.00%)
  💾 band_0.20_0.60: 100000 records
    🔎 Active molecules: 5000/100000 (5.00%)
  💾 band_0.30_0.70: 100000 records
    🔎 Active molecules: 5000/100000 (5.00%)
  💾 band_0.40_0.80: 100000 records
    🔎 Active molecules: 5000/100000 (5.00%)
  💾 band_0.50_0.90: 100000 records
    🔎 Active molecules: 5000/100000 (5.00%)
  💾 band_0.60_1.00: 100000 records
    🔎 Active molecules: 5000/100000 (5.00%)

✅ COMPLETED!
📊 Summary:
  Total input records: 692704
  Records with valid fingerprints: 692704
  Number of bands created: 7
  Total records in all bands: 700000
  Total active molecules across bands: 32039 (4.6%)

✅ (Morgan, KLIFS) pairs have been d